# Error Attribution Execution: L=0.18 CUDA Stress Geometry

Sequential execution of Probes 1–4 against the L=0.18 archived CUDA stress assets.
Each stage has an explicit go/no-go gate. Do not advance to the next stage until the
current gate passes.

**Primary anomaly to explain:** `transient_pearson_r = 0.99748` vs `transient_r2 = -2.6595`.
The FNO mean prediction is worse than using the SPICE trajectory mean as a trivial predictor.

Figures and canonical JSONs go under `DOCS_ROOT` (`docs/assets/.../attribution/`).
Caches, raw probe arrays, and full compose-run trees go under `RUNS_ROOT`
(`runs/attribution/.../`, gitignored). The sidecar `attribution_manifest.json` maps
every probe output back to the baseline `summary.json` so results cannot silently drift.

## Configuration

Edit these paths and geometry parameters before running any cell below.

In [ ]:
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("repo root (pyproject.toml) not found from " + str(start))


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
DOCS_ROOT = REPO_ROOT / "docs" / "assets" / "cs_amp_fno_exp2" / "attribution"
RUNS_ROOT = REPO_ROOT / "runs" / "attribution" / "cs_amp_fno_exp2"


def repo_rel(path: Path) -> str:
    return str(path.resolve().relative_to(REPO_ROOT))


# Geometry — must match the archived stress assets
NFET_W_UM   = 6.0
NFET_L_UM   = 0.18
PFET_W_UM   = 4.5
PFET_L_UM   = 0.18
VIN_BIAS    = 0.85
VDD         = 1.8
DEVICE      = "cuda"


# Compose-run subtrees and raw probe payloads (gitignored `runs/` subtree)
BASELINE_DIR   = RUNS_ROOT / "baseline"
HYBRID_DIR     = RUNS_ROOT / "hybrid"
FULL_SPICE_DIR = RUNS_ROOT / "full_spice_iv"
NR_DIAG_DIR    = RUNS_ROOT / "nr_diag"

# Cache paths (multi-hour builds; keep under RUNS_ROOT)
NFET_CACHE_WEAK  = RUNS_ROOT / "nfet_iv_018_weak.npz"   # weak-inversion band only
PFET_CACHE_WEAK  = RUNS_ROOT / "pfet_iv_018_weak.npz"
NFET_CACHE_FULL  = RUNS_ROOT / "nfet_iv_018_full.npz"   # full range (sanity run)
PFET_CACHE_FULL  = RUNS_ROOT / "pfet_iv_018_full.npz"

# PDK
PDK_ROOT = "/app/sky130_volare"
CORNER   = "tt"

# IV error bad-region threshold (from plan: sky130 NFET Vth ~ 0.42 V)
VGS_BAD_THRESHOLD = 0.50  # tighten after Probe 1 results

DOCS_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
print("Docs root (figures, JSON):", DOCS_ROOT)
print("Runs root (caches, arrays, compose trees):", RUNS_ROOT)

---
## Stage 1 — Baseline compose run

Reproduce the archived stress-geometry numbers. Gate: published metrics must match
within floating-point tolerance before any attribution run starts.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [
        sys.executable, "-m", "spino.circuit.compose",
        "--nfet-w", str(NFET_W_UM), "--nfet-l", str(NFET_L_UM),
        "--pfet-w", str(PFET_W_UM), "--pfet-l", str(PFET_L_UM),
        "--vin-bias", str(VIN_BIAS), "--vdd", str(VDD),
        "--device", DEVICE,
        "--output-dir", str(BASELINE_DIR),
    ],
    check=False, text=True,
)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"baseline compose failed with exit code {result.returncode}")


In [ ]:
import json

# Published reference numbers from docs/results.md
PUBLISHED = {
    "dc_rel_error_vdd":    0.009209,
    "transient_pearson_r": 0.99748,
    "transient_max_dv_mv": 25.78,
    "transient_r2":        -2.6595,
}

summary = json.loads((BASELINE_DIR / "summary.json").read_text())

measured = {
    "dc_rel_error_vdd":    summary["dc_op"]["nominal"]["report"]["rel_v_out_error_vdd"],
    "transient_pearson_r": summary["transient"]["transient_pearson_r"],
    "transient_max_dv_mv": summary["transient"]["transient_max_abs_error_v"] * 1e3,
    "transient_r2":        summary["transient"]["transient_r2"],
}

print(f"{'Metric':<28} {'Published':>12} {'Measured':>12} {'Match':>8}")
print("-" * 64)
gate_ok = True
for key, pub in PUBLISHED.items():
    meas = measured[key]
    # Reproduce within 0.1% relative tolerance
    match = abs(meas - pub) / (abs(pub) + 1e-12) < 0.001
    gate_ok = gate_ok and match
    flag = "OK" if match else "FAIL"
    print(f"{key:<28} {pub:>12.5f} {meas:>12.5f} {flag:>8}")

print()
if gate_ok:
    print("GATE PASSED: baseline reproduces published metrics. Proceed to Stage 2.")
else:
    print("GATE FAILED: baseline does not match. Resolve before continuing.")
    raise RuntimeError("Baseline reproduction failed — do not proceed with attribution.")

---
## Stage 2 — Probe 1: IV error at fixed terminal voltages

Produced by the compose run above (no `--skip-attribution`). Load and check face validity:
IV error must be larger in the bad region (Vgs < threshold) than the good region.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

p1 = np.load(BASELINE_DIR / "attribution" / "probe1_iv_error.npz")
time_ns    = p1["time_s"] * 1e9
err_nfet   = p1["err_nfet_a"]
err_pfet   = p1["err_pfet_a"]

T_STEP_START_NS = 100.0  # matches _DEFAULT_T_STEP_START in compose.py
WINDOW_NS       = 200.0  # comparison window around step transition

transition_mask = (time_ns >= T_STEP_START_NS - WINDOW_NS * 0.5) & \
                  (time_ns <= T_STEP_START_NS + WINDOW_NS * 0.5)
steady_mask     = ~transition_mask

mean_nfet_trans  = err_nfet[transition_mask].mean()
mean_nfet_steady = err_nfet[steady_mask].mean()
mean_pfet_trans  = err_pfet[transition_mask].mean()
mean_pfet_steady = err_pfet[steady_mask].mean()

print(f"NFET  transition mean |ΔId|: {mean_nfet_trans:.3e} A")
print(f"NFET  steady-state mean |ΔId|: {mean_nfet_steady:.3e} A")
print(f"NFET  ratio (transition/steady): {mean_nfet_trans / mean_nfet_steady:.2f}x")
print()
print(f"PFET  transition mean |ΔId|: {mean_pfet_trans:.3e} A")
print(f"PFET  steady-state mean |ΔId|: {mean_pfet_steady:.3e} A")
print(f"PFET  ratio (transition/steady): {mean_pfet_trans / mean_pfet_steady:.2f}x")

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].semilogy(time_ns, err_nfet)
axes[0].axvspan(T_STEP_START_NS - WINDOW_NS * 0.5, T_STEP_START_NS + WINDOW_NS * 0.5,
                alpha=0.15, color="red", label="transition window")
axes[0].axvline(T_STEP_START_NS, color="red", linestyle="--", alpha=0.6)
axes[0].set_ylabel("|ΔId| NFET (A)")
axes[0].set_title("Probe 1: IV error along transient trajectory")
axes[0].legend()

axes[1].semilogy(time_ns, err_pfet)
axes[1].axvspan(T_STEP_START_NS - WINDOW_NS * 0.5, T_STEP_START_NS + WINDOW_NS * 0.5,
                alpha=0.15, color="red", label="transition window")
axes[1].axvline(T_STEP_START_NS, color="red", linestyle="--", alpha=0.6)
axes[1].set_ylabel("|ΔId| PFET (A)")
axes[1].set_xlabel("Time (ns)")
axes[1].legend()

plt.tight_layout()
plt.savefig(DOCS_ROOT / "probe1_iv_error.png", dpi=150)
plt.show()

gate_ok = (mean_nfet_trans > mean_nfet_steady) or (mean_pfet_trans > mean_pfet_steady)
print()
if gate_ok:
    print("GATE PASSED: IV error elevated during step transition. Face validity confirmed.")
else:
    print("GATE FAILED: IV error not concentrated at transition — inspect the plot.")

---
## Stage 2 — Probe 2: KCL residual waveform at pinned SPICE V_out

Must be evaluated pointwise — do not collapse to a scalar. Peak imbalance should
align with the V_in step transition (when V_out traverses low-Vg NFET IV region).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

p2 = json.loads((BASELINE_DIR / "attribution" / "probe2_kcl_residual.json").read_text())

spice_pin = p2["pinned_spice_vout"]
fno_pin   = p2["pinned_fno_vout"]

# KCL tail samples align with interior times t = T_STEP, 2*T_STEP, ... (row 0 is IC, not stored)
T_STEP_S   = 10e-9
n_kcl      = len(spice_pin["waveform_a"])
time_ns    = np.arange(1, n_kcl + 1, dtype=np.float64) * (T_STEP_S * 1e9)

res_spice  = np.array(spice_pin["waveform_a"])
res_fno    = np.array(fno_pin["waveform_a"])

T_STEP_START_NS = 100.0

print("Pinned at SPICE V_out:")
print(f"  IC residual (row 0): {spice_pin['ic_residual_v']:.3e} V")
print(f"  KCL max |i|:          {spice_pin['kcl_max_a']:.3e} A")
print(f"  KCL rms:              {spice_pin['kcl_rms_a']:.3e} A")
print()
print("Pinned at FNO V_out:")
print(f"  IC residual (row 0): {fno_pin['ic_residual_v']:.3e} V")
print(f"  KCL max |i|:          {fno_pin['kcl_max_a']:.3e} A")
print(f"  KCL rms:              {fno_pin['kcl_rms_a']:.3e} A")

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(time_ns, np.array(res_spice) * 1e6)
axes[0].axvline(T_STEP_START_NS, color="red", linestyle="--", alpha=0.6, label="V_in step")
axes[0].set_ylabel("KCL residual (µA)")
axes[0].set_title("Probe 2: KCL residual pinned at SPICE V_out")
axes[0].legend()

axes[1].plot(time_ns, np.array(res_fno) * 1e6)
axes[1].axvline(T_STEP_START_NS, color="red", linestyle="--", alpha=0.6, label="V_in step")
axes[1].set_ylabel("KCL residual (µA)")
axes[1].set_title("Probe 2: KCL residual pinned at FNO V_out")
axes[1].set_xlabel("Time (ns)")
axes[1].legend()

plt.tight_layout()
plt.savefig(DOCS_ROOT / "probe2_kcl_residual_waveform.png", dpi=150)
plt.show()

print()
print("Interpretation:")
print(f"  SPICE-pinned KCL max: {spice_pin['kcl_max_a']:.3e} A — if large, IV mismatch drives imbalance at reference equilibrium")
print(f"  FNO-pinned KCL max:   {fno_pin['kcl_max_a']:.3e} A — if much larger than SPICE-pinned, coupled solve is amplifying the error")
print(f"  Ratio FNO/SPICE:      {fno_pin['kcl_max_a'] / spice_pin['kcl_max_a']:.2f}x")

gate_ok = spice_pin["kcl_max_a"] > 1e-9
print()
if gate_ok:
    print("GATE PASSED: non-trivial KCL residual at SPICE V_out. Proceed to Stage 3.")
else:
    print("GATE FAILED: residual at SPICE V_out is negligible — IV error is not material at reference equilibrium.")

---
## Stage 3 — Build and validate IV caches

Offline, one-time cost. Weak-inversion band cache is used for the hybrid run.
Full-range cache is used for the sanity baseline.

**Gate:** Both caches must pass Step 1b validation before Stage 4 starts.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [
        sys.executable, "/app/scripts/cache_spice_iv.py", "nfet",
        "--out", str(NFET_CACHE_WEAK),
        "--w-um", str(NFET_W_UM), "--l-um", str(NFET_L_UM),
        "--vg-min", "0.0", "--vg-max", "0.6", "--vg-steps", "61",
        "--vd-min", "0.0", "--vd-max", "1.8", "--vd-steps", "61",
        "--pdk-root", PDK_ROOT, "--corner", CORNER,
        "--validate",
    ],
    check=False, text=True,
)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"NFET weak cache build failed with exit code {result.returncode}")


In [ ]:
import subprocess, sys

result = subprocess.run(
    [
        sys.executable, "/app/scripts/cache_spice_iv.py", "pfet",
        "--out", str(PFET_CACHE_WEAK),
        "--w-um", str(PFET_W_UM), "--l-um", str(PFET_L_UM), "--vs-v", str(VDD),
        "--vg-min", "0.0", "--vg-max", "1.8", "--vg-steps", "61",
        "--vd-min", "0.0", "--vd-max", "1.8", "--vd-steps", "61",
        "--pdk-root", PDK_ROOT, "--corner", CORNER,
        "--validate",
    ],
    check=False, text=True,
)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"PFET weak cache build failed with exit code {result.returncode}")


In [ ]:
import subprocess, sys

for kind, out, w, l, extra in [
    ("nfet", NFET_CACHE_FULL, NFET_W_UM, NFET_L_UM, []),
    ("pfet", PFET_CACHE_FULL, PFET_W_UM, PFET_L_UM, ["--vs-v", str(VDD)]),
]:
    result = subprocess.run(
        [
            sys.executable, "/app/scripts/cache_spice_iv.py", kind,
            "--out", str(out),
            "--w-um", str(w), "--l-um", str(l),
            *extra,
            "--vg-min", "0.0", "--vg-max", "1.8", "--vg-steps", "91",
            "--vd-min", "0.0", "--vd-max", "1.8", "--vd-steps", "91",
            "--pdk-root", PDK_ROOT, "--corner", CORNER,
            "--validate",
        ],
        check=False, text=True,
    )
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"{kind} full cache build failed with exit code {result.returncode}")


In [ ]:
def check_validation_json(val_path: Path, label: str) -> bool:
    val = json.loads(val_path.read_text())
    overall = val["pass"]
    records = val["records"]
    failed  = [r for r in records if not r["pass"]]
    print(f"{label}: overall={'PASS' if overall else 'FAIL'}  "
          f"({len(records)} points checked, {len(failed)} failed)")
    if failed:
        for r in failed:
            print(f"    vg={r.get('vg', '?'):.3f}  vd={r.get('vd', '?'):.3f}  "
                  f"id_cache={r.get('id_cache_a', float('nan')):.3e}  "
                  f"id_op={r.get('id_op_a', float('nan')):.3e}  "
                  f"metric={r.get('metric', {})}")
    return overall

def _val_path(cache: Path) -> Path:
    return cache.parent / (cache.stem + "_validation.json")

results = [
    check_validation_json(_val_path(NFET_CACHE_WEAK), "NFET weak cache"),
    check_validation_json(_val_path(PFET_CACHE_WEAK), "PFET weak cache"),
    check_validation_json(_val_path(NFET_CACHE_FULL), "NFET full cache"),
    check_validation_json(_val_path(PFET_CACHE_FULL), "PFET full cache"),
]
print()
if all(results):
    print("GATE PASSED: all caches validated. Proceed to Stage 5.")
else:
    raise RuntimeError("Cache validation failed.")

---
## Stage 4 — Probe 3: Hybrid IV substitution run

Three variants: full FNO (baseline from Stage 1), hybrid (SPICE in bad region),
full SPICE IV (sanity floor). Gate: hybrid metrics must differ from baseline —
identical numbers indicate the wrapping was a no-op.

In [ ]:
# Verify PFET bad region convention is correct
# VSG = Vs - Vg; for PFET in CS amp, Vs = VDD = 1.8V, Vg = V_out (~0.59V at bias)
# So VSG = 1.8 - 0.59 = ~1.21V at the DC operating point
# bad_region_fn receives VSG and returns True when VSG < threshold
# At threshold=0.40: VSG < 0.40 means near-off (gate close to rail), which is correct
# At threshold=99.0: VSG < 99.0 is always True, routes everything to SPICE

vdd_v    = VDD          # 1.8
vg_at_op = 0.5924       # FNO V_out at DC OP from baseline summary.json
vsg_at_op = vdd_v - vg_at_op

print(f"VSG at DC operating point: {vsg_at_op:.4f} V")
print(f"Hybrid threshold:          0.40 V")
print(f"Operating point in bad region (VSG < 0.40)? {vsg_at_op < 0.40}")
print()
print("Expected: False — the DC operating point should be in the GOOD region")
print("If True, the threshold is too aggressive and will route the steady-state")
print("operating point through SPICE, which conflates IV error with the substitution signal.")

In [ ]:
import subprocess, sys

result = subprocess.run(
    [
        sys.executable, "-m", "spino.circuit.compose",
        "--nfet-w", str(NFET_W_UM), "--nfet-l", str(NFET_L_UM),
        "--pfet-w", str(PFET_W_UM), "--pfet-l", str(PFET_L_UM),
        "--vin-bias", str(VIN_BIAS), "--vdd", str(VDD),
        "--device", DEVICE,
        "--output-dir", str(HYBRID_DIR),
        "--hybrid-iv",
        "--nfet-iv-cache", str(NFET_CACHE_FULL),   # use full-range cache now
        "--pfet-iv-cache", str(PFET_CACHE_FULL),   # use full-range cache now
        "--hybrid-vgs-threshold-nfet", "1.0",      # covers 0.85-0.90V operating range
        "--hybrid-vsg-threshold-pfet", "1.5",      # covers 1.19-1.27V VSG operating range
        "--skip-attribution",
    ],
    check=False, text=True,
)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"hybrid compose failed with exit code {result.returncode}")

In [ ]:
# Full SPICE IV sanity run — thresholds cover entire bias range
result = subprocess.run(
    [
        sys.executable, "-m", "spino.circuit.compose",
        "--nfet-w", str(NFET_W_UM), "--nfet-l", str(NFET_L_UM),
        "--pfet-w", str(PFET_W_UM), "--pfet-l", str(PFET_L_UM),
        "--vin-bias", str(VIN_BIAS), "--vdd", str(VDD),
        "--device", DEVICE,
        "--output-dir", str(FULL_SPICE_DIR),
        "--hybrid-iv",
        "--nfet-iv-cache", str(NFET_CACHE_FULL),
        "--pfet-iv-cache", str(PFET_CACHE_FULL),
        "--hybrid-vgs-threshold-nfet", "99.0",
        "--hybrid-vsg-threshold-pfet", "99.0",
        "--skip-attribution",
    ],
    check=False, text=True,
)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"full SPICE IV compose failed with exit code {result.returncode}")

In [ ]:
def load_transient_metrics(summary_path: Path) -> dict:
    s = json.loads(summary_path.read_text())
    t = s["transient"]
    return {
        "pearson_r":    t["transient_pearson_r"],
        "r2":           t["transient_r2"],
        "max_dv_mv":    t["transient_max_abs_error_v"] * 1e3,
        "dc_rel_error": s["dc_op"]["nominal"]["report"]["rel_v_out_error_vdd"],
    }

runs = {
    "Full FNO (baseline)": load_transient_metrics(BASELINE_DIR   / "summary.json"),
    "Hybrid (bad→SPICE)":  load_transient_metrics(HYBRID_DIR     / "summary.json"),
    "Full SPICE IV":        load_transient_metrics(FULL_SPICE_DIR / "summary.json"),
}

print(f"{'Run':<24} {'Pearson r':>10} {'R²':>10} {'max|ΔV| mV':>12} {'DC rel err':>12}")
print("-" * 72)
for label, m in runs.items():
    print(f"{label:<24} {m['pearson_r']:>10.5f} {m['r2']:>10.4f} "
          f"{m['max_dv_mv']:>12.2f} {m['dc_rel_error']:>12.6f}")

baseline_maxdv = runs["Full FNO (baseline)"]["max_dv_mv"]
hybrid_maxdv   = runs["Hybrid (bad→SPICE)"]["max_dv_mv"]
spice_maxdv    = runs["Full SPICE IV"]["max_dv_mv"]
collapse_frac  = (baseline_maxdv - hybrid_maxdv) / baseline_maxdv

print(f"\nError collapse fraction:      {collapse_frac:.3f}  ({baseline_maxdv:.2f} → {hybrid_maxdv:.2f} mV)")
print(f"Full SPICE floor:             {spice_maxdv:.2f} mV")
if (baseline_maxdv - spice_maxdv) != 0:
    print(f"Fraction toward SPICE floor:  {(baseline_maxdv - hybrid_maxdv) / (baseline_maxdv - spice_maxdv):.3f}")

print()
if abs(hybrid_maxdv - baseline_maxdv) < 0.1:
    print("No collapse — bad_mask still not triggering. Check cache coverage vs Vgs range.")
elif collapse_frac > 0.5:
    print("RESULT: Substantial collapse. IV error at the operating point is causal.")
elif collapse_frac > 0.2:
    print("RESULT: Partial collapse. IV error is a contributor; other factors remain.")
else:
    print("RESULT: Minimal collapse. Proceed to Stage 5 (NR diagnostics).")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
labels = list(runs.keys())
maxdv  = [m["max_dv_mv"] for m in runs.values()]
r2vals = [m["r2"]        for m in runs.values()]
colors = ["steelblue", "darkorange", "forestgreen"]

axes[0].bar(labels, maxdv, color=colors)
axes[0].set_ylabel("max |ΔV| (mV)")
axes[0].set_title("Transient max absolute error")
axes[0].tick_params(axis="x", rotation=15)

axes[1].bar(labels, r2vals, color=colors)
axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_ylabel("Transient R²")
axes[1].set_title("Transient R²")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig(DOCS_ROOT / "probe3_three_run_comparison.png", dpi=150)
plt.show()

In [ ]:
print("Stage 4 conclusion:")
print("  Substitution with narrow thresholds (0.50/0.40): bad mask never triggered.")
print("  Transient Vgs = 0.85-0.90V throughout — operating point is not in weak inversion.")
print("  Substitution with broad thresholds (1.0/1.5): NR diverges.")
print("  When all timesteps are bad, FNO Jacobian and SPICE current values are inconsistent.")
print("  Substitution causal closure is not achievable with this NR architecture")
print("  for errors at the nominal operating point.")
print()
print("  Causal evidence from Probe 2 stands independently:")
print(f"  KCL imbalance at SPICE V_out: {spice_pin['kcl_max_a']:.3e} A")
print(f"  KCL imbalance at FNO V_out:   {fno_pin['kcl_max_a']:.3e} A")
print(f"  Ratio: {spice_pin['kcl_max_a'] / fno_pin['kcl_max_a']:.0f}x")
print("  The solver converges correctly to a wrong IV surface. IV error is causal.")

---
## Stage 5 — Probe 4: NR diagnostics (conditional)

Run only if Stage 4 showed minimal or no error collapse. Instruments
`TransientSolver._newton_step` to log Jacobian diagonal ratio, linear
solve residual, and line-search alpha. Does not add `linalg.cond`.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [
        sys.executable, "-m", "spino.circuit.compose",
        "--nfet-w", str(NFET_W_UM), "--nfet-l", str(NFET_L_UM),
        "--pfet-w", str(PFET_W_UM), "--pfet-l", str(PFET_L_UM),
        "--vin-bias", str(VIN_BIAS), "--vdd", str(VDD),
        "--device", DEVICE,
        "--output-dir", str(NR_DIAG_DIR),
        "--nr-diagnostics",
        "--skip-attribution",
    ],
    check=False, text=True,
)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"NR diagnostics compose failed with exit code {result.returncode}")

In [ ]:
nr_log_path = NR_DIAG_DIR / "attribution" / "nr_newton_diagnostics.json"
nr_records  = json.loads(nr_log_path.read_text())

iters         = list(range(len(nr_records)))
diag_ratios   = [r["jacobian_diag_ratio"]        for r in nr_records]
solve_res     = [r["linear_solve_residual_norm"] for r in nr_records]
alphas        = [r["alpha"]                      for r in nr_records]
alpha_min_val = nr_records[0]["alpha_min"] if nr_records else 1e-3

print(f"{'Iter':>6} {'Diag ratio':>14} {'Solve residual':>16} {'Alpha':>10} {'At min?':>8}")
print("-" * 60)
for i, r in enumerate(nr_records):
    at_min = "YES" if abs(r["alpha"] - r["alpha_min"]) < 1e-8 else ""
    print(f"{i:>6} {r['jacobian_diag_ratio']:>14.3e} "
          f"{r['linear_solve_residual_norm']:>16.3e} "
          f"{r['alpha']:>10.4f} {at_min:>8}")

print()
print("Interpretation:")
print("  diag_ratio > 1e6  -> Jacobian ill-conditioned; solver may be contributing.")
print("  solve_residual large -> linear solve is inaccurate; check conditioning.")
print("  alpha at alpha_min   -> line-search stalled; nonconvex FNO surface likely.")
print("  all nominal          -> solver is fine; residual floor is IV accuracy.")

fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)

axes[0].semilogy(iters, diag_ratios, "o-")
axes[0].axhline(1e6, color="red", linestyle="--", alpha=0.5, label="1e6 threshold")
axes[0].set_ylabel("Jacobian diag ratio")
axes[0].set_title("NR Diagnostics — transient solver")
axes[0].legend()

axes[1].semilogy(iters, solve_res, "o-")
axes[1].axhline(1e-6, color="red", linestyle="--", alpha=0.5, label="1e-6 floor")
axes[1].set_ylabel("Linear solve residual")
axes[1].legend()

axes[2].plot(iters, alphas, "o-")
axes[2].axhline(alpha_min_val, color="red", linestyle="--", alpha=0.5,
                label=f"alpha_min = {alpha_min_val}")
axes[2].set_ylabel("Step alpha")
axes[2].set_xlabel("NR iteration")
axes[2].legend()

plt.tight_layout()
plt.savefig(DOCS_ROOT / "probe4_nr_diagnostics.png", dpi=150)
plt.show()

---
## Stage 6 — VTC Probe 1: IV error across the full VTC bias sweep

The VTC sweep covers Vin from 0 to VDD in 0.05V steps. At each bias point we have the
SPICE DC OP V_out from the baseline summary. We build the exact probe tensors the
solvers would see and compare FNO drain current against SPICE branch current at those
terminals.

This is the IV error map across the full VTC — it shows *where* on the transfer curve
the FNO IV surface diverges from SPICE, independent of any coupling effects.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
import torch
from pathlib import Path

baseline  = json.loads((BASELINE_DIR / "summary.json").read_text())
vtc       = baseline["vtc"]
vin_grid  = np.array(vtc["vin_v"])
spice_vtc = np.array(vtc["spice_v_out_v"])
fno_vtc   = np.array(vtc["fno_v_out_v"])

from spino.circuit.composition_io import load_cs_amp_devices
nfet_raw, pfet_raw = load_cs_amp_devices(
    nfet_w_um=NFET_W_UM, nfet_l_um=NFET_L_UM,
    pfet_w_um=PFET_W_UM, pfet_l_um=PFET_L_UM,
    map_location=DEVICE,
)
_dev   = nfet_raw.v_mean.device
_dtype = nfet_raw.v_mean.dtype

from spino.circuit.standalone_mosfet import build_isolated_mosfet_circuit, isolated_mosfet_id_a
from spino.circuit.simulation import run_operating_point

T_PROBE = 256
TRIM    = 16

err_nfet_vtc      = np.zeros(len(vin_grid))
err_pfet_vtc      = np.zeros(len(vin_grid))
spice_id_vtc      = np.zeros(len(vin_grid))    # NFET SPICE Id
spice_id_pfet_vtc = np.zeros(len(vin_grid))    # PFET SPICE Id — was missing before
fno_id_nfet_vtc   = np.zeros(len(vin_grid))
fno_id_pfet_vtc   = np.zeros(len(vin_grid))

print("Running VTC IV error sweep...")
for idx, (vin, vout_spice) in enumerate(zip(vin_grid, spice_vtc)):
    nfet_probe = torch.cat([
        torch.full((1,1,T_PROBE), vin,        dtype=_dtype, device=_dev),
        torch.full((1,1,T_PROBE), vout_spice, dtype=_dtype, device=_dev),
        torch.zeros(1,1,T_PROBE,              dtype=_dtype, device=_dev),
        torch.zeros(1,1,T_PROBE,              dtype=_dtype, device=_dev),
    ], dim=1)
    pfet_probe = torch.cat([
        torch.full((1,1,T_PROBE), vout_spice, dtype=_dtype, device=_dev),
        torch.full((1,1,T_PROBE), vout_spice, dtype=_dtype, device=_dev),
        torch.full((1,1,T_PROBE), VDD,        dtype=_dtype, device=_dev),
        torch.full((1,1,T_PROBE), VDD,        dtype=_dtype, device=_dev),
    ], dim=1)
    with torch.no_grad():
        id_nfet_fno = float(nfet_raw.drain_current(nfet_probe)[..., TRIM:].mean())
        id_pfet_fno = float(pfet_raw.drain_current(pfet_probe)[..., TRIM:].mean())
    ck = build_isolated_mosfet_circuit(
        is_pfet=False, width_um=NFET_W_UM, length_um=NFET_L_UM,
        vg=float(vin), vd=float(vout_spice), vs=0.0, vb=0.0,
        pdk_root=PDK_ROOT, corner=CORNER,
    )
    op = run_operating_point(ck)
    id_nfet_spice = isolated_mosfet_id_a(op) if op else float("nan")
    ck_p = build_isolated_mosfet_circuit(
        is_pfet=True, width_um=PFET_W_UM, length_um=PFET_L_UM,
        vg=float(vout_spice), vd=float(vout_spice), vs=VDD, vb=VDD,
        pdk_root=PDK_ROOT, corner=CORNER,
    )
    op_p = run_operating_point(ck_p)
    # NGSpice i(@m.xiv...[id]) returns positive for PFET drain current
    id_pfet_spice = abs(isolated_mosfet_id_a(op_p)) if op_p else float("nan")
    err_nfet_vtc[idx]      = abs(id_nfet_fno - id_nfet_spice)
    err_pfet_vtc[idx]      = abs(id_pfet_fno - id_pfet_spice)
    spice_id_vtc[idx]      = id_nfet_spice
    spice_id_pfet_vtc[idx] = id_pfet_spice
    fno_id_nfet_vtc[idx]   = id_nfet_fno
    fno_id_pfet_vtc[idx]   = id_pfet_fno
    if idx % 5 == 0:
        print(f"  Vin={vin:.2f}V  NFET err={err_nfet_vtc[idx]:.2e}A  PFET err={err_pfet_vtc[idx]:.2e}A")

print("Done.")

In [ ]:
VTC_BAD_THRESHOLD = 0.50  # Vin below this = weak inversion for NFET; near-off for PFET

np.savez(
    DOCS_ROOT / "vtc_probe1_iv_error.npz",
    vin=vin_grid,
    err_nfet_a=err_nfet_vtc,
    err_pfet_a=err_pfet_vtc,
    spice_id_nfet_a=spice_id_vtc,
    spice_id_pfet_a=spice_id_pfet_vtc,
    fno_id_nfet_a=fno_id_nfet_vtc,
    fno_id_pfet_a=fno_id_pfet_vtc,
)

REL_FLOOR = 1e-12
rel_err_nfet = err_nfet_vtc / np.maximum(np.abs(spice_id_vtc),       REL_FLOOR)
rel_err_pfet = err_pfet_vtc / np.maximum(np.abs(spice_id_pfet_vtc),  REL_FLOOR)

# Cap PFET at 1000x — the near-off spike at Vin~0.25V (FNO predicts ~13.7A vs 785pA)
# is a real catastrophic failure but its magnitude dominates the mean uninformatively
rel_err_pfet_capped = np.minimum(rel_err_pfet, 1000.0)

# Both devices fail in the SAME Vin < threshold region:
# NFET: weak inversion (low Vgs)
# PFET: near-off (low VSG = VDD - Vout, because Vout is high when Vin is low)
bad  = vin_grid < VTC_BAD_THRESHOLD
good = ~bad

fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

axes[0].semilogy(vin_grid, rel_err_nfet,        "o-", markersize=4, label="NFET relative |ΔId|")
axes[0].semilogy(vin_grid, rel_err_pfet_capped, "s-", markersize=4, label="PFET relative |ΔId| (capped 1000x)")
axes[0].axvline(VTC_BAD_THRESHOLD, color="red", linestyle="--", alpha=0.6,
                label=f"Vin threshold = {VTC_BAD_THRESHOLD}V")
axes[0].set_ylabel("Relative |ΔId|")
axes[0].set_title("VTC Probe 1: relative IV error across VTC sweep")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].semilogy(vin_grid, err_nfet_vtc, "o-", markersize=4, label="NFET absolute |ΔId|")
axes[1].semilogy(vin_grid, err_pfet_vtc, "s-", markersize=4, label="PFET absolute |ΔId|")
axes[1].axvline(VTC_BAD_THRESHOLD, color="red", linestyle="--", alpha=0.6)
axes[1].set_ylabel("|ΔId| (A)")
axes[1].set_title("VTC Probe 1: absolute IV error across VTC sweep")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(vin_grid, spice_vtc, color="#0066cc", linewidth=1.6, label="SPICE V_out")
axes[2].plot(vin_grid, fno_vtc,   color="#cc6600", linewidth=1.2, linestyle="--", label="FNO V_out")
axes[2].axvline(VTC_BAD_THRESHOLD, color="red", linestyle="--", alpha=0.6)
axes[2].set_ylabel("V_out (V)")
axes[2].set_xlabel("V_in (V)")
axes[2].set_title("VTC overlay (reference)")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(DOCS_ROOT / "vtc_probe1_iv_error.png", dpi=150)
plt.show()

print(f"NFET bad  region (Vin < {VTC_BAD_THRESHOLD}V): mean relative |ΔId| = {rel_err_nfet[bad].mean():.3f}")
print(f"NFET good region (Vin >= {VTC_BAD_THRESHOLD}V): mean relative |ΔId| = {rel_err_nfet[good].mean():.3f}")
print(f"NFET ratio bad/good: {rel_err_nfet[bad].mean() / rel_err_nfet[good].mean():.1f}x")
print()
print(f"PFET bad  region (Vin < {VTC_BAD_THRESHOLD}V): mean relative |ΔId| (capped) = {rel_err_pfet_capped[bad].mean():.3f}")
print(f"PFET good region (Vin >= {VTC_BAD_THRESHOLD}V): mean relative |ΔId| (capped) = {rel_err_pfet_capped[good].mean():.3f}")
print(f"PFET ratio bad/good: {rel_err_pfet_capped[bad].mean() / (rel_err_pfet_capped[good].mean() + 1e-12):.1f}x")
print()
print("Note: PFET spike at Vin~0.25V (VSG~0.33V): FNO predicts ~13.7A vs SPICE 785pA.")
print("      Near-off subthreshold failure — same root cause as NFET, more severe.")

gate_ok = (
    rel_err_nfet[bad].mean() > rel_err_nfet[good].mean() and
    rel_err_pfet_capped[bad].mean() > rel_err_pfet_capped[good].mean()
)
print()
if gate_ok:
    print("GATE PASSED: both devices show elevated relative IV error in the low-Vin region.")
    print("Proceed to Stage 7.")
else:
    nfet_ok = rel_err_nfet[bad].mean() > rel_err_nfet[good].mean()
    pfet_ok = rel_err_pfet_capped[bad].mean() > rel_err_pfet_capped[good].mean()
    print(f"GATE FAILED: NFET={'PASS' if nfet_ok else 'FAIL'}  PFET={'PASS' if pfet_ok else 'FAIL'}")
    print("Inspect relative error plot before continuing.")

---
## Stage 7 — VTC DC OP substitution (causal closure)

Run `DcOperatingPointSolver` at each VTC bias point with the hybrid device (SPICE IV
in the bad region). Compare the hybrid V_out against the full-FNO V_out and SPICE
reference. If the hybrid collapses toward SPICE in the bad region, IV error at those
bias points is causal.

This avoids the NR Jacobian consistency problem that blocked the transient
substitution: each DC solve is independent and scalar, so there is no trajectory-wide
Jacobian to corrupt.

In [ ]:
# Stage 7 — Cell 1 (replace entirely)
from scipy.optimize import brentq
from scipy.interpolate import RegularGridInterpolator
import numpy as np

# Load caches
nfet_cache = np.load(NFET_CACHE_WEAK)
pfet_cache = np.load(PFET_CACHE_WEAK)

nfet_interp = RegularGridInterpolator(
    (nfet_cache["vg"], nfet_cache["vd"]),
    nfet_cache["ids"], method="linear",
    bounds_error=False, fill_value=None,
)
pfet_interp = RegularGridInterpolator(
    (pfet_cache["vg"], pfet_cache["vd"]),
    pfet_cache["ids"], method="linear",
    bounds_error=False, fill_value=None,
)

def kcl_residual_spice(v_out, vin):
    """KCL residual using SPICE IV cache for both devices."""
    i_nfet = nfet_interp([[vin, v_out]])[0]
    i_pfet = pfet_interp([[v_out, v_out]])[0]
    return float(i_pfet - i_nfet)

from spino.circuit.composition import DcOperatingPointSolver

dc_fno = DcOperatingPointSolver(nfet_raw, pfet_raw, vdd=VDD)

vout_fno_sweep     = np.zeros(len(vin_grid))
vout_hybrid_sweep  = np.zeros(len(vin_grid))

print("Running VTC DC OP substitution sweep...")
v_seed_fno = VDD / 2.0

for idx, (vin, vout_spice) in enumerate(zip(vin_grid, spice_vtc)):
    sol_fno = dc_fno.solve(vin=float(vin), v_out_init=v_seed_fno)
    vout_fno_sweep[idx] = sol_fno.v_out_v
    v_seed_fno = sol_fno.v_out_v

    if float(vin) < VTC_BAD_THRESHOLD:
        # Bad region: solve KCL directly using SPICE cache via brentq
        try:
            fa = kcl_residual_spice(0.0, float(vin))
            fb = kcl_residual_spice(VDD, float(vin))
            if fa * fb < 0:
                vout_hybrid_sweep[idx] = brentq(
                    kcl_residual_spice, 0.0, VDD,
                    args=(float(vin),), xtol=1e-6, maxiter=100,
                )
            else:
                vout_hybrid_sweep[idx] = vout_fno_sweep[idx]
                print(f"  WARNING: no bracket at Vin={vin:.2f}V — using FNO fallback")
        except ValueError:
            vout_hybrid_sweep[idx] = vout_fno_sweep[idx]
            print(f"  WARNING: brentq failed at Vin={vin:.2f}V — using FNO fallback")
    else:
        # Good region: pure FNO — cache not valid here
        vout_hybrid_sweep[idx] = vout_fno_sweep[idx]

    if idx % 5 == 0:
        print(f"  Vin={vin:.2f}V  FNO={vout_fno_sweep[idx]:.4f}V  "
              f"Hybrid={vout_hybrid_sweep[idx]:.4f}V  SPICE={spice_vtc[idx]:.4f}V")

np.savez(
    DOCS_ROOT / "vtc_substitution.npz",
    vin=vin_grid,
    vout_fno=vout_fno_sweep,
    vout_hybrid=vout_hybrid_sweep,
    vout_spice=spice_vtc,
)
print("Done.")

In [ ]:
# Compute collapse fractions and plot
err_fno_vtc    = np.abs(vout_fno_sweep    - spice_vtc)
err_hybrid_vtc = np.abs(vout_hybrid_sweep - spice_vtc)

bad  = vin_grid < VTC_BAD_THRESHOLD
good = ~bad

collapse_bad  = (err_fno_vtc[bad].mean()  - err_hybrid_vtc[bad].mean())  / (err_fno_vtc[bad].mean()  + 1e-12)
collapse_good = (err_fno_vtc[good].mean() - err_hybrid_vtc[good].mean()) / (err_fno_vtc[good].mean() + 1e-12)

print(f"Bad region  (Vin < {VTC_BAD_THRESHOLD}V):")
print(f"  FNO mean |ΔVout|:    {err_fno_vtc[bad].mean()*1e3:.2f} mV")
print(f"  Hybrid mean |ΔVout|: {err_hybrid_vtc[bad].mean()*1e3:.2f} mV")
print(f"  Collapse fraction:   {collapse_bad:.3f}")

max_fno_bad     = err_fno_vtc[bad].max()
max_hybrid_bad  = err_hybrid_vtc[bad].max()
collapse_bad_max = (max_fno_bad - max_hybrid_bad) / (max_fno_bad + 1e-12)
print(f"  FNO max  |ΔVout|:    {max_fno_bad*1e3:.2f} mV")
print(f"  Hybrid max  |ΔVout|: {max_hybrid_bad*1e3:.2f} mV")
print(f"  Max-collapse fraction: {collapse_bad_max:.3f}")
print()
print(f"Good region (Vin >= {VTC_BAD_THRESHOLD}V):")
print(f"  FNO mean |ΔVout|:    {err_fno_vtc[good].mean()*1e3:.2f} mV")
print(f"  Hybrid mean |ΔVout|: {err_hybrid_vtc[good].mean()*1e3:.2f} mV")
print(f"  Collapse fraction:   {collapse_good:.3f}")

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

axes[0].plot(vin_grid, spice_vtc,         color="#0066cc", linewidth=1.6, label="SPICE")
axes[0].plot(vin_grid, vout_fno_sweep,    color="#cc6600", linewidth=1.2, linestyle="--", label="Full FNO")
axes[0].plot(vin_grid, vout_hybrid_sweep, color="#009933", linewidth=1.2, linestyle=":",  label="Hybrid (SPICE IV in bad region)")
axes[0].axvline(VTC_BAD_THRESHOLD, color="red", linestyle="--", alpha=0.5)
axes[0].set_ylabel("V_out (V)")
axes[0].set_title("VTC substitution: three-way comparison")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(vin_grid, err_fno_vtc    * 1e3, color="#cc6600", linewidth=1.2, linestyle="--", label="Full FNO |ΔVout|")
axes[1].plot(vin_grid, err_hybrid_vtc * 1e3, color="#009933", linewidth=1.2, linestyle=":",  label="Hybrid |ΔVout|")
axes[1].axvline(VTC_BAD_THRESHOLD, color="red", linestyle="--", alpha=0.5, label=f"Vin threshold = {VTC_BAD_THRESHOLD}V")
axes[1].set_ylabel("|ΔVout| (mV)")
axes[1].set_xlabel("V_in (V)")
axes[1].set_title("VTC error: FNO vs Hybrid")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(DOCS_ROOT / "vtc_substitution.png", dpi=150)
plt.show()

gate_ok = collapse_bad > 0.3
print()
if not gate_ok:
    print("GATE FAILED: hybrid does not collapse VTC error in bad region.")
    print("Check cache coverage vs VTC bias range (Vin 0.0-0.45V).")
elif collapse_bad > 0.7:
    print("RESULT: Strong collapse in bad region. Weak-inversion IV error is causal for VTC failure.")
elif collapse_bad > 0.3:
    print("RESULT: Partial collapse. IV error is a primary contributor; other factors remain.")
print(f"  Good-region collapse: {collapse_good:.3f} (should be near zero — confirms specificity)")

---
## Stage 8 — VTC attribution summary

Consolidate VTC findings alongside the transient attribution. Populate the VTC section
of the attribution result and update the manifest.

In [ ]:
# Final three-way VTC comparison figure for docs
vtc_sub_npz   = np.load(DOCS_ROOT / "vtc_substitution.npz")
vin_arr       = vtc_sub_npz["vin"]
bad_mask_vtc  = vin_arr < VTC_BAD_THRESHOLD

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(vin_arr, vtc_sub_npz["vout_spice"],  color="#0066cc", linewidth=1.6, label="SPICE")
axes[0].plot(vin_arr, vtc_sub_npz["vout_fno"],    color="#cc6600", linewidth=1.2, linestyle="--", label="Full FNO")
axes[0].plot(vin_arr, vtc_sub_npz["vout_hybrid"], color="#009933", linewidth=1.2, linestyle=":",  label="Hybrid")
axes[0].axvline(VTC_BAD_THRESHOLD, color="red", linestyle="--", alpha=0.5, label="Threshold")
axes[0].set_xlabel("V_in (V)")
axes[0].set_ylabel("V_out (V)")
axes[0].set_title("VTC: FNO vs Hybrid vs SPICE")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

labels  = ["Full FNO\n(bad region)", "Hybrid\n(bad region)", "Full FNO\n(good region)", "Hybrid\n(good region)"]
heights = [
    err_fno_vtc[bad].mean()    * 1e3,
    err_hybrid_vtc[bad].mean() * 1e3,
    err_fno_vtc[good].mean()   * 1e3,
    err_hybrid_vtc[good].mean()* 1e3,
]
colors_bar = ["#cc6600", "#009933", "#cc6600", "#009933"]
bars = axes[1].bar(labels, heights, color=colors_bar, alpha=0.85)
axes[1].set_ylabel("Mean |ΔVout| (mV)")
axes[1].set_title("VTC error: bad vs good region")
axes[1].grid(True, alpha=0.3, axis="y")
for bar, h in zip(bars, heights):
    axes[1].text(bar.get_x() + bar.get_width()/2, h + 0.5, f"{h:.1f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(DOCS_ROOT / "vtc_attribution_summary.png", dpi=150)
plt.show()

print("Saved vtc_attribution_summary.png")


---
## Stage 9 — Deliverable summary and manifest

Populate the final attribution JSON and verify all outputs are present.
This is the original Stage 6, pushed to the end after VTC attribution is complete.

In [ ]:
import json
import numpy as np
from pathlib import Path
from scipy.stats import pearsonr

baseline     = json.loads((BASELINE_DIR / "summary.json").read_text())
p1_tran      = np.load(BASELINE_DIR / "attribution" / "probe1_iv_error.npz")
p2           = json.loads((BASELINE_DIR / "attribution" / "probe2_kcl_residual.json").read_text())
spice_pin    = p2["pinned_spice_vout"]
fno_pin      = p2["pinned_fno_vout"]
nr_records   = json.loads((NR_DIAG_DIR / "attribution" / "nr_newton_diagnostics.json").read_text())
vtc_p1       = np.load(DOCS_ROOT / "vtc_probe1_iv_error.npz")
vtc_sub      = np.load(DOCS_ROOT / "vtc_substitution.npz")

vin_arr      = vtc_sub["vin"]
bad_mask_vtc = vin_arr < VTC_BAD_THRESHOLD
err_fno_arr  = np.abs(vtc_sub["vout_fno"]    - vtc_sub["vout_spice"])
err_hyb_arr  = np.abs(vtc_sub["vout_hybrid"] - vtc_sub["vout_spice"])

collapse_bad_mean = float(
    (err_fno_arr[bad_mask_vtc].mean() - err_hyb_arr[bad_mask_vtc].mean())
    / (err_fno_arr[bad_mask_vtc].mean() + 1e-12)
)
collapse_bad_max = float(
    (err_fno_arr[bad_mask_vtc].max() - err_hyb_arr[bad_mask_vtc].max())
    / (err_fno_arr[bad_mask_vtc].max() + 1e-12)
)

# Bad-region Pearson r: V_out shape fidelity in the failure band.
vtc_pearson_r_full, _    = pearsonr(vtc_sub["vout_spice"],         vtc_sub["vout_fno"])
vtc_pearson_r_bad_fno, _ = pearsonr(vtc_sub["vout_spice"][bad_mask_vtc], vtc_sub["vout_fno"][bad_mask_vtc])
vtc_pearson_r_bad_hyb, _ = pearsonr(vtc_sub["vout_spice"][bad_mask_vtc], vtc_sub["vout_hybrid"][bad_mask_vtc])

REL_FLOOR        = 1e-12
spice_id_nfet    = vtc_p1["spice_id_nfet_a"]
spice_id_pfet    = vtc_p1["spice_id_pfet_a"]
fno_id_pfet      = vtc_p1["fno_id_pfet_a"]
rel_err_nfet     = vtc_p1["err_nfet_a"] / np.maximum(np.abs(spice_id_nfet), REL_FLOOR)
rel_err_pfet_raw = vtc_p1["err_pfet_a"] / np.maximum(np.abs(spice_id_pfet), REL_FLOOR)
rel_err_pfet_cap = np.minimum(rel_err_pfet_raw, 1000.0)

pfet_bad_median  = float(np.median(rel_err_pfet_raw[bad_mask_vtc]))
pfet_good_median = float(np.median(rel_err_pfet_raw[~bad_mask_vtc]))
pfet_bad_geomean = float(np.exp(np.log(np.maximum(rel_err_pfet_raw[bad_mask_vtc], REL_FLOOR)).mean()))
pfet_good_geomean = float(np.exp(np.log(np.maximum(rel_err_pfet_raw[~bad_mask_vtc], REL_FLOOR)).mean()))

# Locate the catastrophic PFET near-off spike (max |I_FNO - I_SPICE| in the bad band).
bad_idx_in_arr = np.where(bad_mask_vtc)[0]
spike_local    = int(np.argmax(vtc_p1["err_pfet_a"][bad_mask_vtc]))
spike_idx      = int(bad_idx_in_arr[spike_local])
pfet_spike_vin       = float(vin_arr[spike_idx])
pfet_spike_fno_a     = float(fno_id_pfet[spike_idx])
pfet_spike_spice_a   = float(spice_id_pfet[spike_idx])

attribution_result = {
    "metadata": {
        "geometry":       "L=0.18 CUDA stress",
        "nfet_w_um":      NFET_W_UM,
        "nfet_l_um":      NFET_L_UM,
        "pfet_w_um":      PFET_W_UM,
        "pfet_l_um":      PFET_L_UM,
        "vdd_v":          VDD,
        "vin_bias_v":     VIN_BIAS,
        "device":         DEVICE,
        "baseline_run":   repo_rel(BASELINE_DIR / "summary.json"),
    },
    "baseline_metrics": {
        "dc_rel_error_vdd":    baseline["dc_op"]["nominal"]["report"]["rel_v_out_error_vdd"],
        "transient_pearson_r": baseline["transient"]["transient_pearson_r"],
        "transient_max_dv_mv": baseline["transient"]["transient_max_abs_error_v"] * 1e3,
        "transient_r2":        baseline["transient"]["transient_r2"],
        "vtc_pearson_r":       baseline["vtc"]["pearson_r"],
        "vtc_max_abs_error_mv":baseline["vtc"]["max_abs_error_v"] * 1e3,
    },
    "transient_attribution": {
        "probe_1_iv_error": {
            "isolates":        "FNO vs SPICE drain current along transient trajectory (Vgs=0.85-0.90V)",
            "mean_nfet_err_a": float(p1_tran["err_nfet_a"].mean()),
            "max_nfet_err_a":  float(p1_tran["err_nfet_a"].max()),
            "mean_pfet_err_a": float(p1_tran["err_pfet_a"].mean()),
            "max_pfet_err_a":  float(p1_tran["err_pfet_a"].max()),
            "note":            "Transient Vgs=0.85-0.90V; weak-inversion locus not traversed by step stimulus",
        },
        "probe_2_kcl_residual": {
            "isolates":             "KCL imbalance at pinned SPICE V_out vs pinned FNO V_out",
            "spice_pinned_kcl_max_a":   spice_pin["kcl_max_a"],
            "spice_pinned_kcl_rms_a":   spice_pin["kcl_rms_a"],
            "fno_pinned_kcl_max_a":     fno_pin["kcl_max_a"],
            "fno_pinned_kcl_rms_a":     fno_pin["kcl_rms_a"],
            "ratio_spice_over_fno": round(spice_pin["kcl_max_a"] / fno_pin["kcl_max_a"], 1),
        },
        "probe_3_substitution": {
            "outcome": "Not executable",
            "reason":  (
                "Transient Vgs=0.85-0.90V throughout; narrow threshold (0.50V) never triggers. "
                "Broad threshold causes NR divergence: FNO Jacobian and SPICE current values "
                "are inconsistent when all timesteps route to cache. "
                "Probe 2 KCL ratio (268x) is the causal evidence."
            ),
        },
        "probe_4_nr_diagnostics": {
            "iterations":         len(nr_records),
            "max_diag_ratio":     max(r["jacobian_diag_ratio"]        for r in nr_records),
            "min_solve_residual": min(r["linear_solve_residual_norm"] for r in nr_records),
            "alpha_always_1":     all(r["alpha"] == 1.0               for r in nr_records),
            "alpha_min":          nr_records[0]["alpha_min"],
        },
        "conclusion": (
            "NR is stable on this trajectory (alpha=1.0 every iteration, 3 iterations). "
            "The Stage-5 Jacobian diag ratio (~616) is a cheap max(|diag|)/min(|diag|) proxy, not cond(J); "
            "the healthy-solver claim rests on alpha and iteration count, not that ratio alone. "
            "19.6uA KCL imbalance at SPICE V_out vs 73nA at FNO V_out (268x ratio). "
            "FNO converges correctly to wrong IV surface at nominal bias (Vgs~0.85V). "
            "Numerics do not contribute."
        ),
    },
    "vtc_attribution": {
        "probe_1_iv_error": {
            "isolates":                 "FNO vs SPICE drain current at each VTC bias point (fixed SPICE terminals)",
            "nfet_ratio_bad_over_good":  round(float(rel_err_nfet[bad_mask_vtc].mean() / rel_err_nfet[~bad_mask_vtc].mean()), 1),
            "pfet_ratio_bad_over_good_mean_capped": round(float(rel_err_pfet_cap[bad_mask_vtc].mean() / (rel_err_pfet_cap[~bad_mask_vtc].mean() + 1e-12)), 1),
            "pfet_median_rel_err_bad":   pfet_bad_median,
            "pfet_median_rel_err_good":  pfet_good_median,
            "pfet_geomean_rel_err_bad":  pfet_bad_geomean,
            "pfet_geomean_rel_err_good": pfet_good_geomean,
            "pfet_near_off_spike": {
                "vin_v":     pfet_spike_vin,
                "fno_id_a":   pfet_spike_fno_a,
                "spice_id_a": pfet_spike_spice_a,
                "note":      "Catastrophic near-off failure: FNO predicts ~A-scale drain current vs SPICE ~pA at VSG~0.33V. Values reproduce from vtc_probe1_iv_error.npz alongside this JSON.",
            },
            "pfet_robustness_note":      "PFET mean-capped ratio reflects a per-point 1000x clip; median and geomean over the raw relative-Id are the robust alternatives.",
        },
        "substitution": {
            "isolates":               "brentq KCL solve on SPICE IV cache in bad region; FNO in good region",
            "threshold_vin_v":        VTC_BAD_THRESHOLD,
            "fno_mean_err_bad_mv":    float(err_fno_arr[bad_mask_vtc].mean()  * 1e3),
            "hybrid_mean_err_bad_mv": float(err_hyb_arr[bad_mask_vtc].mean() * 1e3),
            "fno_max_err_bad_mv":     float(err_fno_arr[bad_mask_vtc].max()  * 1e3),
            "hybrid_max_err_bad_mv":  float(err_hyb_arr[bad_mask_vtc].max()  * 1e3),
            "collapse_fraction_bad_mean": round(collapse_bad_mean, 3),
            "collapse_fraction_bad_max":  round(collapse_bad_max, 3),
            "fno_mean_err_good_mv":   float(err_fno_arr[~bad_mask_vtc].mean() * 1e3),
            "good_region_gate_invariant": True,
            "good_region_note":       "Good region (Vin >= threshold) is never routed to the SPICE cache by construction; hybrid V_out is byte-identical to FNO V_out there. The hybrid_mean_err_good_mv / collapse_fraction_good_mean fields from earlier schema have been dropped as they were structurally zero rather than independent measurements.",
            "vtc_pearson_r_full":     float(vtc_pearson_r_full),
            "vtc_pearson_r_bad_fno":  float(vtc_pearson_r_bad_fno),
            "vtc_pearson_r_bad_hybrid": float(vtc_pearson_r_bad_hyb),
            "pearson_r_note":         "Full-sweep Pearson r masks the bad-region shape collapse. Restricting to Vin < threshold drops FNO r from ~0.99 to ~0.87 and the hybrid recovers it to ~1.00, mirroring the |DeltaV_out| collapse.",
        },
        "conclusion": (
            f"Bad-region V_out error collapses by {collapse_bad_mean*100:.1f}% on the mean "
            f"({err_fno_arr[bad_mask_vtc].mean()*1e3:.1f} -> {err_hyb_arr[bad_mask_vtc].mean()*1e3:.2f} mV) "
            f"and {collapse_bad_max*100:.1f}% on the max "
            f"({err_fno_arr[bad_mask_vtc].max()*1e3:.1f} -> {err_hyb_arr[bad_mask_vtc].max()*1e3:.2f} mV) "
            f"under brentq KCL substitution. Bad-region Pearson r over V_out also collapses: "
            f"FNO {vtc_pearson_r_bad_fno:.3f} vs hybrid {vtc_pearson_r_bad_hyb:.3f} (full-sweep r masks this at {vtc_pearson_r_full:.4f}). "
            f"Good-region error is unchanged by construction (gate routes only bad-region points to the cache). "
            f"NFET shows a {round(float(rel_err_nfet[bad_mask_vtc].mean() / rel_err_nfet[~bad_mask_vtc].mean()), 0):.0f}x bad/good relative-Id ratio; "
            f"PFET median bad-region relative-Id error is {pfet_bad_median:.1f} vs good {pfet_good_median:.3f} (mean-capped ratio is a 1000x clip artefact); "
            f"the PFET near-off spike at Vin={pfet_spike_vin:.2f} V is FNO {pfet_spike_fno_a:+.3g} A vs SPICE {pfet_spike_spice_a:+.3g} A. "
            f"Weak-inversion IV error is causal for VTC failure with strong specificity."
        ),
    },
}

out_path = DOCS_ROOT / "attribution_result.json"
out_path.write_text(json.dumps(attribution_result, indent=2))
print("Wrote:", out_path)
print()
print(json.dumps(attribution_result, indent=2))


In [ ]:
manifest = {
    "baseline_summary":            repo_rel(BASELINE_DIR   / "summary.json"),
    "probe_1_transient_iv_error":  repo_rel(BASELINE_DIR   / "attribution" / "probe1_iv_error.npz"),
    "probe_1_transient_summary":   repo_rel(BASELINE_DIR   / "attribution" / "probe1_iv_error_summary.json"),
    "probe_2_kcl_residual":        repo_rel(BASELINE_DIR   / "attribution" / "probe2_kcl_residual.json"),
    "probe_4_nr_diagnostics":      repo_rel(NR_DIAG_DIR    / "attribution" / "nr_newton_diagnostics.json"),
    "vtc_probe_1_iv_error":        repo_rel(DOCS_ROOT      / "vtc_probe1_iv_error.npz"),
    "vtc_substitution":            repo_rel(DOCS_ROOT      / "vtc_substitution.npz"),
    "vtc_probe1_figure":           repo_rel(DOCS_ROOT      / "vtc_probe1_iv_error.png"),
    "vtc_substitution_figure":     repo_rel(DOCS_ROOT      / "vtc_attribution_summary.png"),
    "probe3_hybrid_summary":       repo_rel(HYBRID_DIR     / "summary.json"),
    "probe3_full_spice_summary":   repo_rel(FULL_SPICE_DIR / "summary.json"),
    "attribution_result":          repo_rel(DOCS_ROOT      / "attribution_result.json"),
}

manifest_path = DOCS_ROOT / "attribution_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print("Manifest written to:", manifest_path)
print()

all_present = True
for key, path_str in manifest.items():
    exists = Path(path_str).exists()
    all_present = all_present and exists
    status = "OK" if exists else "MISSING"
    print(f"  {status:>8}  {key}: {path_str}")

print()
if all_present:
    print("All manifest entries present. Attribution is complete.")
else:
    print("Some manifest entries are missing. Check failed stages above.")